In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import unsloth # Moved to top as per warning
import torch

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

from trl import SFTTrainer
from datasets import load_dataset
from google.colab import userdata
from transformers import TextStreamer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
def format_squad(example):
    context = example["context"]
    question = example["question"]
    answers = example["answers"]["text"]

    answer = answers[0] if len(answers) > 0 else "No answer."

    return {"text": f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a helpful AI assistant providing accurate answers based on the given context.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Context: {context}
Question: {question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer: {answer}
<|eot_id|>"""}

dataset = load_dataset("squad", split = "train")
dataset = dataset.map(format_squad)

In [ ]:
print(dataset[0]["text"])

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 6,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>",
    response_part = "<|start_header_id|>assistant<|end_header_id|>",
)

In [ ]:
# In ra chuỗi token dạng số (máy tính đọc)
print(trainer.train_dataset[0]["input_ids"])

# Dịch ngược từ số ra chữ để con người kiểm tra
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
print(tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]]))

In [ ]:
trainer_stats = trainer.train()

In [ ]:
def format_test_squad(example):
    context = example["context"]
    question = example["question"]

    return {"text": f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a helpful AI assistant providing accurate answers based on the given context.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Context: {context}
Question: {question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer: """}

test_dataset = load_dataset("squad", split = "validation")
test_dataset = test_dataset.map(format_test_squad)

test_dataset

In [ ]:
print(test_dataset[0]['text'])

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer(
    test_dataset[0]['text'],
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs["input_ids"], streamer = text_streamer, max_new_tokens = 64,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
save_path = "MinhQuy24/llama3.2_3B_SQuAD_QA"
HF_KEY=''
model.push_to_hub(save_path, token = HF_KEY)
tokenizer.push_to_hub(save_path, token = HF_KEY)

In [ ]:
# max_seq_length = 2048
# dtype = None
# load_in_4bit = True

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = save_path,
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = load_in_4bit,
# )
# FastLanguageModel.for_inference(model)